In [3]:
"""
pick_best_results.py
====================
Reads all 7 pruning method JSONs and identifies the BEST result per method.

"Best" = the result with the HIGHEST accuracy whose accuracy drop from
baseline does not exceed MAX_ACC_DROP_THRESHOLD (default 1%).

If NO result meets the threshold, falls back to the result with the
SMALLEST accuracy drop (least-worst). This is clearly flagged.

For methods with sub-variants (M3 magnitude, M7 one-shot), the best result
is picked ACROSS all variants at all sparsity levels.

Outputs:
  best_results_summary.json   -- one entry per method with the winning config
  comparison_table.json       -- clean 8-row table (baseline + 7 methods)

Usage:
  python pick_best_results.py        (run from pruning_scripts/ directory)
"""

import json, os

# ── Config ─────────────────────────────────────────────────────────────────
MAX_ACC_DROP_THRESHOLD = 0.01   # 1% -- change to 0.02 for 2%

JSON_FILES = {
    "M1_unstructured"   : "method1_unstructured_pruning.json",
    "M2_structured"     : "method2_structured_pruning.json",
    "M3_magnitude"      : "method3_magnitude_pruning.json",
    "M4_movement"       : "method4_movement_pruning.json",
    "M5_lottery_ticket" : "method5_lottery_ticket_pruning.json",
    "M6_iterative"      : "method6_iterative_pruning.json",
    "M7_oneshot"        : "method7_oneshot_pruning.json",
}

METHOD_LABELS = {
    "M1_unstructured"   : "Unstructured Pruning",
    "M2_structured"     : "Structured Pruning",
    "M3_magnitude"      : "Magnitude Pruning",
    "M4_movement"       : "Movement Pruning",
    "M5_lottery_ticket" : "Lottery Ticket Hypothesis",
    "M6_iterative"      : "Iterative Pruning",
    "M7_oneshot"        : "One-Shot Pruning",
}

# ── Helpers ────────────────────────────────────────────────────────────────
def load_json(path):
    with open(path) as f:
        return json.load(f)

def get_baseline_fields(baseline):
    """Normalize baseline dict -- handles different field name versions."""
    params   = baseline.get("params", baseline.get("num_parameters", 0))
    size_mb  = baseline.get("size_mb", baseline.get("model_size_mb", 0))
    inf_ms   = baseline.get("inference_ms", baseline.get("inference_time_ms", 0))
    if "flops_G" in baseline:
        flops = int(baseline["flops_G"] * 1e9)
    elif "flops" in baseline:
        flops = int(baseline["flops"])
    elif "macs_G" in baseline:
        flops = int(baseline["macs_G"] * 2 * 1e9)
    else:
        flops = 0
    return params, size_mb, inf_ms, flops

def get_config_label(row):
    """Build a human-readable config string from a result row."""
    parts = []
    for key in ["sparsity", "filter_pruning_ratio", "criterion",
                "variant", "round", "cumulative_sparsity"]:
        if key in row:
            val = row[key]
            if isinstance(val, float) and val <= 1.0:
                parts.append(f"{key}={val*100:.0f}%")
            else:
                parts.append(f"{key}={val}")
    return " | ".join(parts) if parts else "—"

def pick_best(results, base_acc, threshold):
    """
    Returns (best_row, passed_threshold).
    1. Prefer rows where (base_acc - row_acc) <= threshold, pick highest acc.
    2. If none pass, pick row with smallest drop (least-worst), flag it.
    """
    scored = [(base_acc - r["accuracy"], r) for r in results]
    within = [(d, r) for d, r in scored if d <= threshold]
    if within:
        _, best = min(within, key=lambda x: x[0])   # smallest drop = highest acc
        return best, True
    else:
        _, best = min(scored, key=lambda x: x[0])
        return best, False


# ── Main ───────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  pick_best_results.py")
    print(f"  Threshold : accuracy drop <= {MAX_ACC_DROP_THRESHOLD*100:.1f}% from baseline")
    print(f"{'='*65}\n")

    summary   = []
    not_found = []
    base_ref  = None   # baseline dict, grabbed from first available file

    for method_id, filename in JSON_FILES.items():
        if not os.path.exists(filename):
            print(f"  [MISSING] {filename} -- skipping {method_id}")
            not_found.append(method_id)
            continue

        data     = load_json(filename)
        baseline = data["baseline"]
        results  = data["results"]
        base_acc = baseline["accuracy"]

        if base_ref is None:
            base_ref = baseline

        best_row, passed = pick_best(results, base_acc, MAX_ACC_DROP_THRESHOLD)
        acc_drop         = base_acc - best_row["accuracy"]
        config_label     = get_config_label(best_row)
        _, base_size, _, _ = get_baseline_fields(baseline)

        entry = {
            "method_id"          : method_id,
            "method_label"       : METHOD_LABELS[method_id],
            "passed_threshold"   : passed,
            "threshold_used"     : MAX_ACC_DROP_THRESHOLD,
            "config"             : config_label,
            # 8 standardized metrics
            "accuracy"           : round(best_row["accuracy"],           6),
            "precision"          : round(best_row["precision"],          6),
            "recall"             : round(best_row["recall"],             6),
            "f1"                 : round(best_row["f1"],                 6),
            "num_parameters"     : best_row["num_parameters"],
            "model_size_mb"      : round(best_row["model_size_mb"],      4),
            "inference_time_ms"  : round(best_row["inference_time_ms"],  4),
            "flops"              : best_row["flops"],
            # derived
            "accuracy_drop"      : round(acc_drop,          6),
            "accuracy_drop_pct"  : round(acc_drop * 100,    4),
            "size_reduction_pct" : round(
                (1 - best_row["model_size_mb"] / base_size) * 100, 2
            ) if base_size else 0,
            # keep references
            "full_best_row"      : best_row,
            "baseline"           : baseline,
        }
        summary.append(entry)

        status = "within threshold" if passed else "FALLBACK -- all configs exceed threshold"
        print(f"  {method_id}")
        print(f"    config  : {config_label}")
        print(f"    accuracy: {best_row['accuracy']:.4f}  drop: {acc_drop*100:+.3f}%  [{status}]")
        print()

    # ── Save best_results_summary.json ────────────────────────────────────
    output = {"threshold": MAX_ACC_DROP_THRESHOLD, "methods": summary}
    with open("best_results_summary.json", "w") as f:
        json.dump(output, f, indent=2)
    print(f"  Saved --> best_results_summary.json\n")

    # ── Build comparison_table.json ────────────────────────────────────────
    if summary and base_ref:
        b_params, b_size, b_inf, b_flops = get_baseline_fields(base_ref)

        table_rows = [{
            "method"            : "Baseline (ResNet-50)",
            "config"            : "no pruning",
            "accuracy"          : round(base_ref["accuracy"],  6),
            "precision"         : round(base_ref["precision"], 6),
            "recall"            : round(base_ref["recall"],    6),
            "f1"                : round(base_ref["f1"],        6),
            "num_parameters"    : b_params,
            "model_size_mb"     : round(b_size, 4),
            "inference_time_ms" : round(b_inf,  4),
            "flops"             : b_flops,
            "accuracy_drop_pct" : 0.0,
            "size_reduction_pct": 0.0,
            "passed_threshold"  : True,
        }]

        for e in summary:
            table_rows.append({
                "method"            : e["method_label"],
                "config"            : e["config"],
                "accuracy"          : e["accuracy"],
                "precision"         : e["precision"],
                "recall"            : e["recall"],
                "f1"                : e["f1"],
                "num_parameters"    : e["num_parameters"],
                "model_size_mb"     : e["model_size_mb"],
                "inference_time_ms" : e["inference_time_ms"],
                "flops"             : e["flops"],
                "accuracy_drop_pct" : e["accuracy_drop_pct"],
                "size_reduction_pct": e["size_reduction_pct"],
                "passed_threshold"  : e["passed_threshold"],
            })

        with open("comparison_table.json", "w") as f:
            json.dump(table_rows, f, indent=2)
        print(f"  Saved --> comparison_table.json\n")

    # ── Print console report table ─────────────────────────────────────────
    print(f"\n{'='*75}")
    print("  FINAL COMPARISON TABLE  (best result per method)")
    print(f"{'='*75}")
    hdr = (f"  {'Method':<30} {'Accuracy':>9} {'Drop':>8} "
           f"{'F1':>8} {'Size MB':>9} {'Params M':>9} {'FLOPs G':>9}")
    print(hdr)
    print("  " + "-"*72)

    if base_ref:
        b_params, b_size, b_inf, b_flops = get_baseline_fields(base_ref)
        print(f"  {'Baseline':<30} {base_ref['accuracy']:>9.4f} {'—':>8} "
              f"{base_ref['f1']:>8.4f} {b_size:>9.2f} {b_params/1e6:>9.2f} "
              f"{b_flops/1e9:>9.3f}")

    for e in summary:
        flag = "" if e["passed_threshold"] else "  (*)"
        print(f"  {e['method_label']:<30} {e['accuracy']:>9.4f} "
              f"{e['accuracy_drop_pct']:>+7.3f}% {e['f1']:>8.4f} "
              f"{e['model_size_mb']:>9.2f} {e['num_parameters']/1e6:>9.2f} "
              f"{e['flops']/1e9:>9.3f}{flag}")

    print(f"\n  (*) No config met the {MAX_ACC_DROP_THRESHOLD*100:.0f}% threshold -- showing least-worst config")
    if not_found:
        print(f"  Missing files (skipped): {not_found}")
    print(f"{'='*75}\n")


if __name__ == "__main__":
    main()


  pick_best_results.py
  Threshold : accuracy drop <= 1.0% from baseline

  M1_unstructured
    config  : sparsity=30%
    accuracy: 0.9321  drop: -0.010%  [within threshold]

  M2_structured
    config  : filter_pruning_ratio=10%
    accuracy: 0.9322  drop: -0.020%  [within threshold]

  M3_magnitude
    config  : sparsity=30% | criterion=global_l1
    accuracy: 0.9321  drop: -0.010%  [within threshold]

  M4_movement
    config  : sparsity=30%
    accuracy: 0.1000  drop: +83.200%  [FALLBACK -- all configs exceed threshold]

  M5_lottery_ticket
    config  : sparsity=30%
    accuracy: 0.9321  drop: -0.010%  [within threshold]

  M6_iterative
    config  : round=6 | cumulative_sparsity=62%
    accuracy: 0.9322  drop: -0.020%  [within threshold]

  M7_oneshot
    config  : sparsity=30% | variant=one_shot_l1
    accuracy: 0.9321  drop: -0.010%  [within threshold]

  Saved --> best_results_summary.json

  Saved --> comparison_table.json


  FINAL COMPARISON TABLE  (best result per method